# 03 — Pemeriksaan submission akhir (010 dan 012)

Notebook ini **memeriksa dua CSV yang sudah disubmit**, bukan melatih model baru. Hasil utama tetap submission 010 (public Macro F1 0,60627); submission 012 (0,60449) disimpan sebagai eksperimen lanjutan. Angka public berasal dari laporan unggahan Kaggle tanggal 13–14 Oktober 2026, bukan perhitungan label test tersembunyi atau pembacaan langsung Kaggle API.

Pemeriksaan berikut meliputi SHA-256, jumlah baris, label 1–5, urutan building_id terhadap sample resmi, perubahan kelas 010→012, dan distribusi prediksi. File asli tidak akan ditulis ulang.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from IPython.display import display
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "submissions").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "submissions").is_dir():
    raise FileNotFoundError("Jalankan notebook dari root repository atau folder notebooks.")

data_zip = ROOT / "data" / "structural-damage-classification.zip"
with ZipFile(data_zip) as archive:
    members = {Path(name).name.lower(): name for name in archive.namelist()}
    assert {"sample_submission.csv", "test.csv"} <= set(members)
    sample = pd.read_csv(archive.open(members["sample_submission.csv"]), dtype={"building_id": str})
    test_ids = pd.read_csv(archive.open(members["test.csv"]), usecols=["building_id"], dtype={"building_id": str})

assert sample.columns.tolist() == ["building_id", "damage_grade"]
assert sample["building_id"].notna().all() and sample["building_id"].is_unique
assert test_ids["building_id"].notna().all() and test_ids["building_id"].is_unique
assert len(sample) == len(test_ids) == 228629
assert set(sample["building_id"]) == set(test_ids["building_id"])

history = json.loads((ROOT / "reports" / "submission10_followup_results.json").read_text(encoding="utf-8"))
metrics12 = json.loads((ROOT / "reports" / "fixed_run" / "metrics.json").read_text(encoding="utf-8"))
print("Sample dan test sesuai:", len(sample), "building_id unik.")
print("Notebook hanya membaca CSV submission dan data uji; tidak melakukan fit model.")

In [ ]:
specs = {
    "010": {
        "file": "010_Constantine(10).csv",
        "sha256": history["submission10"]["sha256"],
        "reported_public_macro_f1": history["submission10"]["public_macro_f1"],
    },
    "012": {
        "file": "012_Constantine_leakage_fixed.csv",
        "sha256": metrics12["submission_sha256"],
        "reported_public_macro_f1": metrics12["public_kaggle_score"],
    },
}
frames, checks = {}, {}
for version, spec in specs.items():
    path = ROOT / "submissions" / spec["file"]
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert digest == spec["sha256"], f"SHA-256 tidak sesuai: {path.name}"

    frame = pd.read_csv(path, dtype={"building_id": str})
    assert frame.columns.tolist() == ["building_id", "damage_grade"]
    assert len(frame) == 228629 and frame["building_id"].is_unique
    assert frame["building_id"].notna().all()
    assert frame["building_id"].equals(sample["building_id"]), f"Urutan sample berbeda: {version}"
    assert frame["damage_grade"].notna().all()
    assert frame["damage_grade"].astype(str).str.fullmatch("[1-5]").all()
    frame["damage_grade"] = frame["damage_grade"].astype(int)
    frames[version] = frame
    checks[version] = {
        "filename": path.name,
        "sha256": digest,
        "rows": len(frame),
        "sample_order_identical": True,
        "grade_counts": {str(k): int(v) for k, v in frame["damage_grade"].value_counts().sort_index().items()},
        "public_macro_f1_user_report": float(spec["reported_public_macro_f1"]),
        "private_macro_f1": None,
    }

assert checks["010"]["public_macro_f1_user_report"] == 0.60627
assert checks["012"]["public_macro_f1_user_report"] == 0.60449
display(pd.DataFrame.from_dict(checks, orient="index")[["filename", "rows", "public_macro_f1_user_report", "sha256"]])
print("Kedua CSV lolos pemeriksaan SHA, skema, label, dan urutan ID.")

## Bagaimana prediksi 010 dan 012 berbeda?

Tabel transisi berikut membandingkan **prediksi dengan prediksi**, bukan kelas aktual dengan prediksi. Oleh karena itu tabel ini **bukan confusion matrix** dan tidak bisa digunakan menghitung F1 pada hidden test.

In [ ]:
transition = pd.crosstab(
    frames["010"]["damage_grade"],
    frames["012"]["damage_grade"],
    rownames=["Prediksi 010"],
    colnames=["Prediksi 012"],
).reindex(index=range(1, 6), columns=range(1, 6), fill_value=0)
changed = int((frames["010"]["damage_grade"] != frames["012"]["damage_grade"]).sum())
assert int(transition.to_numpy().sum()) == len(sample)
assert changed == metrics12["changed_vs_submission10"] == 20831

print(f"Berbeda: {changed:,} dari {len(sample):,} baris ({100 * changed / len(sample):.2f}%).")
display(transition)
counts = pd.DataFrame({
    "010": frames["010"]["damage_grade"].value_counts().reindex(range(1, 6), fill_value=0),
    "012": frames["012"]["damage_grade"].value_counts().reindex(range(1, 6), fill_value=0),
})
counts.index.name = "Predicted grade"
counts["012 minus 010"] = counts["012"] - counts["010"]
display(counts)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(transition.to_numpy(), cmap="Blues")
for i in range(5):
    for j in range(5):
        v = int(transition.iloc[i, j])
        ax.text(j, i, f"{v:,}", ha="center", va="center",
                fontsize=8, color="white" if v > transition.to_numpy().max() * 0.5 else "black")
ax.set(xticks=range(5), yticks=range(5),
       xticklabels=range(1, 6), yticklabels=range(1, 6),
       xlabel="Prediksi 012", ylabel="Prediksi 010",
       title="Transisi prediksi 010 ke 012 (bukan confusion matrix)")
fig.colorbar(im, ax=ax, label="Jumlah bangunan")
fig.tight_layout()
review_dir = ROOT / "artifacts" / "submission_review"
review_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(review_dir / "prediction_transitions_010_012.png", dpi=140)
plt.show()

review = {
    "kind": "prediction_integrity_and_comparison_not_test_label_evaluation",
    "versions": checks,
    "changed_predictions": changed,
    "changed_share": changed / len(sample),
    "reported_public_delta_012_minus_010": round(0.60449 - 0.60627, 5),
    "transition_010_to_012": transition.to_numpy().tolist(),
    "hidden_test_labels_used": False,
    "final_recommendation": "010",
    "note": "Skor public berasal dari laporan peserta; bukti Kaggle yang menghubungkan unggahan ke checksum belum diarsipkan.",
}
(review_dir / "submission_010_012_review.json").write_text(
    json.dumps(review, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Laporan audit:", review_dir / "submission_010_012_review.json")

## Interpretasi

Sebagian prediksi berubah antara 010 dan 012, tetapi jumlah perubahan tidak membuktikan peningkatan akurasi. Berdasarkan public leaderboard yang dilaporkan, skor 012 (0,60449) masih di bawah 010 (0,60627) sebesar 0,00178. Karena itu **010 dipertahankan sebagai submission utama** dan 012 sebagai pembanding.

Batas verifikasi: CSV 010 dapat dicek integritasnya, tetapi training ensemble historis yang menghasilkan 010 secara persis **belum direproduksi**. Eksperimen 012 memiliki dokumentasi audit lokal terpisah. Skor private belum tersedia dan tidak ditaksir dari isi CSV. Jika kompetisi mengizinkan dua final submission, 010 dan 012 dapat dipilih; keputusan akhirnya tetap mengikuti aturan kompetisi.